# 03 · DKA cohort and Path C (schema authored in Arrows)

Tests whether a schema drawn in Neo4j Arrows, exported as JSON and converted to prompt text, answers the same question bank as well as the hand-written schema (Path A).

**Arms:** Path A (hand-written) · Path C (Arrows JSON) · Path C+ (Arrows JSON + prose notes). Same Text2Cypher prompt, validator, graph and 34-question bank; only the schema text differs.

**Needs:** a PhysioNet-credentialed Google account (MIMIC-IV on BigQuery), an Aura Free instance, an Anthropic API key. Keys are typed at prompts and never stored in this notebook.

> Credentialed MIMIC-IV derived output must not be committed. This notebook is saved with outputs cleared; keep it that way (Colab: Edit → Clear all outputs, or `jupyter nbconvert --clear-output`).

## 1 · Setup

In [ ]:
!pip install -q neo4j anthropic pandas google-cloud-bigquery db-dtypes pyarrow
from google.colab import drive, files
drive.mount('/content/drive')

In [ ]:
# Upload: dka_build.py, arrows.py, dka_questions.py, your Arrows JSON export, and the Aura credentials .txt
up = files.upload()
print(list(up))

In [ ]:
exec(open('/content/dka_build.py').read())
exec(open('/content/arrows.py').read())
exec(open('/content/dka_questions.py').read())

import glob
txt = glob.glob('/content/*.txt')[0]
cfg = dict(l.strip().split('=', 1) for l in open(txt) if '=' in l)
run, drv = connect(cfg['NEO4J_URI'], cfg['NEO4J_USERNAME'], cfg['NEO4J_PASSWORD'])
del cfg   # do not keep the password in memory longer than needed
verify(run)

## 2 · Build the DKA graph (only if the Aura instance is empty)

Cohort: 96 patients with ≥3 admissions whose principal diagnosis is diabetic ketoacidosis. Expected: 1,979 admissions, 552 DKA admissions. Skip this section if `verify(run)` above already shows those counts.

In [ ]:
!gcloud auth application-default login --no-launch-browser
PROJECT = "<your-gcp-project-id>"   # quota project; the Google account must have PhysioNet MIMIC-IV access
!gcloud auth application-default set-quota-project {PROJECT}

In [ ]:
dfs = fetch(project=PROJECT)     # ~2 min
build(dfs, run)                  # ~6 min
verify(run)

## 3 · Schemas

`SCHEMA` is the hand-written string in `dka_build.py` (Path A). Path C reads the Arrows export. `validate()` compares the drawing against the live graph and prints any drift (labels, properties, relationships, in both directions).

In [ ]:
ARROWS = '/content/schema_dka.arrows.json'   # your exported file
validate(run, ARROWS)
SCHEMA_B = from_arrows(ARROWS)                 # Path C
SCHEMA_D = from_arrows(ARROWS, notes=NOTES)    # Path C+
print(SCHEMA_B)

## 4 · Question bank (34 questions: simple 8, medium 8, hard 8, DKA 6, adversarial 4)

In [ ]:
bank = build_bank(run)           # resolves H02, H07, X03 against the live graph
gold = verify_gold(bank, run)    # runs every reference query; no API calls

## 5 · Run the arms

Each arm is run and saved separately, so an interruption costs at most one arm. About $1 per arm at 3 runs. Model `claude-opus-5`, no temperature set.

In [ ]:
import getpass, time, anthropic, pandas as pd, pickle, os
aclient = anthropic.Anthropic(api_key=getpass.getpass("Anthropic API key: "))

def llm(prompt, max_tokens=2000):
    for attempt in range(4):
        try:
            r = aclient.messages.create(model="claude-opus-5", max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}])
            # the response may begin with a thinking block; keep only text blocks
            return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        except Exception:
            if attempt == 3: raise
            time.sleep(2 * (attempt + 1))

In [ ]:
OUT = '/content/drive/MyDrive/graphrag_clinical/dka'   # personal Drive, NOT the repo
os.makedirs(OUT, exist_ok=True)
arms = {"Path A (hand-written)": SCHEMA, "Path C (Arrows)": SCHEMA_B,
        "Path C+ (Arrows + notes)": SCHEMA_D}
done = {}
for name, sch in arms.items():
    done.update(run_all_arms({name: sch}, bank, run, n_runs=3))
    pickle.dump(done, open(f'{OUT}/allr.pkl', 'wb'))
    print("saved", name)
allr = pd.concat(done.values(), ignore_index=True)
report(allr)

## 6 · Per-question view
Questions that score below 100 in any arm, as % of runs passed.

In [ ]:
allr["arm"] = allr["arm"].str.replace(r" \(run \d+\)$", "", regex=True)
t = allr.pivot_table(index="id", columns="arm", values="projection", aggfunc="mean") * 100
print(t[(t < 100).any(axis=1)].round(0))
print(allr.groupby("arm").projection.agg(["sum", "count", "mean"]).round(3))